In [1]:
%pip install chess


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import torch, pickle
from pathlib import Path
import sys
sys.path.append(str((Path.cwd().parent / "nanochat").resolve()))
from nanochat.gpt import GPT, GPTConfig
import chess, chess.pgn

In [ ]:
ckpt = torch.load("../models/chess_min_gpu.pt", map_location="cpu")
config = GPTConfig(**ckpt["meta"]["model_config"])
model = GPT(config).eval()
model.load_state_dict(ckpt["model"])
stoi = ckpt["meta"]["tokenizer"]["stoi"]
itos = ckpt["meta"]["tokenizer"]["itos"]

In [4]:
prompt_text = "<bos> d4 d5"
prompt_tokens = prompt_text.strip().split()
prompt_ids = [stoi[token] for token in prompt_tokens]

In [11]:
board = chess.Board()

def reset_board_from_tokens(tokens):
    board.reset()
    for token in tokens:
        if token in {"<bos>", "<eos>"}:
            continue
        board.push_san(token)

def sample_next_token(logits, board, stoi, itos, forbid_eos=False):
    legal_san = {board.san(mv) for mv in board.legal_moves}
    mask = torch.full((len(itos),), float("-inf"), device=logits.device)
    # this mask is a simple hack to prevent the sampling of invalid moves & early eos
    # sets all the tokens I don't want to sample to -inf, then softmaxing & sampling
    # will only sample from the legal moves

    for token, idx in stoi.items():
        if token == "<bos>":
            continue
        elif token == "<eos>":
            if forbid_eos:
                continue
            mask[idx] = logits[idx]
        elif token in legal_san:
            mask[idx] = logits[idx]
    filtered = torch.softmax(mask, dim=-1)
    next_id = torch.multinomial(filtered, num_samples=1)
    next_token = itos[next_id.item()]
    if next_token not in {"<bos>", "<eos>"}:
        board.push_san(next_token)
    return next_id, next_token
# board

In [12]:
prompt_san = [itos[idx] for idx in prompt_ids]
reset_board_from_tokens(prompt_san)

In [13]:
device = next(model.parameters()).device
x = torch.tensor(prompt_ids, device=device)[None, :]
generated = prompt_san[:]

eos_cnt = 0
max_eos_cnt = 5
for _ in range(200):
    logits = model(x[:, -config.sequence_len:])
    next_id, next_token = sample_next_token(logits[:, -1, :].squeeze(0), board, stoi, itos, forbid_eos=eos_cnt >= max_eos_cnt)
    x = torch.cat([x, next_id.view(1, 1)], dim=1)
    generated.append(next_token)
    if next_token == "<eos>":
        eos_cnt += 1
        if eos_cnt >= max_eos_cnt:
            break

In [14]:
game = chess.pgn.Game()
node = game
board = game.board()
for token in generated:
    if token in {"<bos>", "<eos>"}:
        continue
    move = board.parse_san(token)
    node = node.add_variation(move)
    board.push(move)
exporter = chess.pgn.StringExporter(headers=False, variations=False, comments=False)
print(game.accept(exporter))

1. d4 d5 2. c4 dxc4 3. Qa4+ Bd7 4. Qxc4 Nc6 5. Nc3 e5 6. Nf3 Nf6 7. Bg5 Bd6 8.
dxe5 Nxe5 9. Qd5 Nxf3+ 10. exf3 Rf8 11. f4 h6 12. Ne4 Bc6 13. Nc3 Bb4 14. O-O-O
Bxc3 15. bxc3 a5 16. Qd3 b5 17. Be2 a4 18. f5 Rb8 19. f4 b4 20. Kb2 a3+ 21. Kb1
b3 22. axb3 Rb5 23. Ka1 Qe7 24. Bf3 Rb8 25. Qe2 Rb7 26. Rc1 Ba4 27. Rb1 Bb5 28.
c4 Ba4 29. bxa4 c6 30. Qc2 g6 31. Qd2 c5 32. Bg4 Ne4 33. Qd5 Ng3 34. Qd8+ *


In [45]:
# use https://lichess.org/paste to run the pgn